In [1]:
import pandas as pd

df = pd.read_csv("../data/home-credit-default-risk/application_train.csv")

In [2]:
core_features = [
    "CODE_GENDER", "CNT_CHILDREN", "NAME_FAMILY_STATUS",
    "CNT_FAM_MEMBERS", "NAME_EDUCATION_TYPE", "DAYS_BIRTH",
    "NAME_INCOME_TYPE", "OCCUPATION_TYPE", "ORGANIZATION_TYPE",
    "DAYS_EMPLOYED", "AMT_INCOME_TOTAL",
    "FLAG_OWN_CAR", "OWN_CAR_AGE", "FLAG_OWN_REALTY",
    "NAME_HOUSING_TYPE", "DAYS_REGISTRATION", "DAYS_ID_PUBLISH",
    "NAME_CONTRACT_TYPE", "AMT_CREDIT", "AMT_ANNUITY",
    "AMT_GOODS_PRICE", "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3",
    "AMT_REQ_CREDIT_BUREAU_HOUR", "AMT_REQ_CREDIT_BUREAU_DAY",
    "AMT_REQ_CREDIT_BUREAU_WEEK", "AMT_REQ_CREDIT_BUREAU_MON",
    "AMT_REQ_CREDIT_BUREAU_QRT", "AMT_REQ_CREDIT_BUREAU_YEAR",
    "TARGET"
]

In [3]:
import numpy as np

df = df[core_features].copy()

df["AGE"] = -df["DAYS_BIRTH"] / 365
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)
df["EMPLOYMENT_YEARS"] = -df["DAYS_EMPLOYED"] / 365

df["DTI_PROXY"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
df["LOAN_TO_INCOME"] = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]

df = df.drop(columns=["DAYS_BIRTH"])

In [4]:
X = df.drop(columns=["TARGET"])
y = df["TARGET"]

## Split and stratify

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

## Distinguish numeric and categorical columns

In [6]:
numeric_features = [
    "CNT_CHILDREN", "CNT_FAM_MEMBERS", "AMT_INCOME_TOTAL",
    "OWN_CAR_AGE", "DAYS_EMPLOYED", "DAYS_REGISTRATION",
    "DAYS_ID_PUBLISH", "AMT_CREDIT", "AMT_ANNUITY",
    "AMT_GOODS_PRICE", "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3",
    "AMT_REQ_CREDIT_BUREAU_HOUR", "AMT_REQ_CREDIT_BUREAU_DAY",
    "AMT_REQ_CREDIT_BUREAU_WEEK", "AMT_REQ_CREDIT_BUREAU_MON",
    "AMT_REQ_CREDIT_BUREAU_QRT", "AMT_REQ_CREDIT_BUREAU_YEAR",
    "AGE", "EMPLOYMENT_YEARS", "DTI_PROXY", "LOAN_TO_INCOME"
]

categorical_features = [
    "CODE_GENDER", "NAME_FAMILY_STATUS", "NAME_EDUCATION_TYPE",
    "NAME_INCOME_TYPE", "OCCUPATION_TYPE", "ORGANIZATION_TYPE",
    "FLAG_OWN_CAR", "FLAG_OWN_REALTY", "NAME_HOUSING_TYPE",
    "NAME_CONTRACT_TYPE"
]

## Preprocessing Pipeline

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [9]:
X_train_processed = preprocessor.fit_transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)
X_test_processed = preprocessor.transform(X_test)

print("Processed train shape:", X_train_processed.shape)
print("Processed validation shape:", X_valid_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed train shape: (215257, 133)
Processed validation shape: (46127, 133)
Processed test shape: (46127, 133)
